In [1]:
import h3
import geopandas as gpd
import pandas as pd
import numpy as np
import json
import folium
from shapely.geometry import Polygon, mapping
from shapely.ops import unary_union

import warnings
warnings.filterwarnings("ignore")

print(f"H3 version: {h3.__version__}")

H3 version: 4.4.2


In [2]:
gdf_pois = gpd.read_file("../data/raw/barcelona_grocery_pois.geojson")
gdf_barris = gpd.read_file("../data/raw/barcelona_barris.geojson")

# Ensure both are WGS84
gdf_pois = gdf_pois.to_crs("EPSG:4326")
gdf_barris = gdf_barris.to_crs("EPSG:4326")

print(f"POIs loaded: {len(gdf_pois)}")
print(f"Barris loaded: {len(gdf_barris)}")
print(f"Barris columns: {gdf_barris.columns.tolist()}")

POIs loaded: 3133
Barris loaded: 73
Barris columns: ['ID_ANNEX', 'ANNEXDESCR', 'ID_TEMA', 'TEMA_DESCR', 'ID_CONJUNT', 'CONJ_DESCR', 'ID_SUBCONJ', 'SCONJ_DESC', 'ID_ELEMENT', 'ELEM_DESCR', 'NIVELL', 'NDESCR_CA', 'NDESCR_ES', 'NDESCR_EN', 'TERME', 'DISTRICTE', 'BARRI', 'AEB', 'SEC_CENS', 'GRANBARRI', 'ZUA', 'AREA_I', 'LITERAL', 'PERIMETRE', 'AREA', 'ORD_REPRES', 'CODI_UA', 'TIPUS_UA', 'NOM', 'WEB1', 'WEB2', 'WEB3', 'DOCUMENTA', 'RANGESCALA', 'TIPUS_POL', 'GRUIX_ID', 'GRUIXDIMEN', 'ESTIL_ID', 'ESTIL_QGIS', 'VALOR1QGIS', 'VALOR2QGIS', 'COL_FARCIT', 'FCOL_DESCR', 'FHEX_COLOR', 'COL_DESCR', 'HEX_COLOR7', 'geometry']


In [3]:
# Dissolve all barris into one Barcelona boundary polygon
barcelona_boundary = gdf_barris.geometry.unary_union

# Spatial join — keep only POIs inside the city boundary
gdf_pois_clipped = gdf_pois[gdf_pois.geometry.within(barcelona_boundary)].copy()
gdf_pois_clipped = gdf_pois_clipped.reset_index(drop=True)

print(f"POIs before clipping: {len(gdf_pois)}")
print(f"POIs after clipping:  {len(gdf_pois_clipped)}")
print(f"Removed {len(gdf_pois) - len(gdf_pois_clipped)} out-of-boundary POIs")

POIs before clipping: 3133
POIs after clipping:  2560
Removed 573 out-of-boundary POIs


In [4]:
RESOLUTION = 9

# Get all H3 hexagons covering the Barcelona boundary
barcelona_geojson = mapping(barcelona_boundary)
hex_ids = list(h3.geo_to_cells(barcelona_geojson, RESOLUTION))

print(f"Resolution {RESOLUTION} → {len(hex_ids)} hexagons covering Barcelona")
print(f"Approx hex diameter: ~{h3.average_hexagon_edge_length(RESOLUTION, unit='m') * 2:.0f}m")

Resolution 9 → 997 hexagons covering Barcelona
Approx hex diameter: ~402m


In [5]:
def assign_h3(row, resolution=RESOLUTION):
    return h3.latlng_to_cell(row.geometry.y, row.geometry.x, resolution)

gdf_pois_clipped["h3_index"] = gdf_pois_clipped.apply(assign_h3, axis=1)

print(f"Unique hexes containing at least 1 store: {gdf_pois_clipped['h3_index'].nunique()}")
print(f"Total hexes in Barcelona: {len(hex_ids)}")
print(f"Coverage: {gdf_pois_clipped['h3_index'].nunique() / len(hex_ids) * 100:.1f}% of hexes have a store")
gdf_pois_clipped[["name", "shop_type", "lat", "lon", "h3_index"]].head(10)

Unique hexes containing at least 1 store: 432
Total hexes in Barcelona: 997
Coverage: 43.3% of hexes have a store


,name,shop_type,lat,lon,h3_index
0,IKEA Swedish Food Market,convenience,41.394893,2.152061,89394460387ffff
1,Open Cor,supermarket,41.391002,2.125651,89394460287ffff
2,Supermarket,convenience,41.412370,2.154647,893944601d7ffff
3,Mercat Felip II,convenience,41.422194,2.185435,89394460c93ffff
4,Condis,supermarket,41.446813,2.174794,89394462853ffff
5,Consum,supermarket,41.441604,2.178063,8939446284fffff
6,Mercadona,supermarket,41.437570,2.180496,89394462ab3ffff
7,Forn de Pa Laia,bakery,41.398263,2.183140,89394460ebbffff
8,Mercadona,supermarket,41.376111,2.131445,89394460267ffff
9,Carniseria Moliner,butcher,41.376989,2.131993,89394460267ffff


In [6]:
# Count stores per hex and by type
hex_counts = (
    gdf_pois_clipped
    .groupby("h3_index")
    .agg(
        store_count   = ("osm_id", "count"),
        shop_types    = ("shop_type", lambda x: list(x.unique())),
        type_diversity= ("shop_type", "nunique"),
        store_names   = ("name", lambda x: list(x))
    )
    .reset_index()
)

print(f"Hexes with at least 1 store: {len(hex_counts)}")
print(hex_counts["store_count"].describe())

Hexes with at least 1 store: 432
count    432.000000
mean       5.925926
std        5.099390
min        1.000000
25%        2.000000
50%        4.000000
75%        8.000000
max       37.000000
Name: store_count, dtype: float64


In [7]:
# For each hex in Barcelona, count stores in k=2 ring (~500m radius)
K = 2

catchment_data = []

for hex_id in hex_ids:
    # Get all hexes within k=2 rings
    ring = h3.grid_disk(hex_id, K)
    
    # Count stores in catchment
    stores_in_catchment = gdf_pois_clipped[
        gdf_pois_clipped["h3_index"].isin(ring)
    ]
    
    catchment_data.append({
        "h3_index":          hex_id,
        "catchment_store_count":  len(stores_in_catchment),
        "catchment_type_diversity": stores_in_catchment["shop_type"].nunique() if len(stores_in_catchment) > 0 else 0,
        "is_food_desert":    len(stores_in_catchment) == 0
    })

df_catchment = pd.DataFrame(catchment_data)

print(f"Total hexes: {len(df_catchment)}")
print(f"Food desert hexes (0 stores in k=2 ring): {df_catchment['is_food_desert'].sum()} ({df_catchment['is_food_desert'].mean()*100:.1f}%)")

Total hexes: 997
Food desert hexes (0 stores in k=2 ring): 234 (23.5%)


In [8]:
def h3_to_polygon(hex_id):
    coords = h3.cell_to_boundary(hex_id)
    # h3 returns (lat, lon) — shapely needs (lon, lat)
    return Polygon([(lon, lat) for lat, lon in coords])

df_catchment["geometry"] = df_catchment["h3_index"].apply(h3_to_polygon)

gdf_hex = gpd.GeoDataFrame(df_catchment, geometry="geometry", crs="EPSG:4326")

# Merge in store-level detail for hexes that have stores
gdf_hex = gdf_hex.merge(hex_counts, on="h3_index", how="left")
gdf_hex["store_count"]    = gdf_hex["store_count"].fillna(0).astype(int)
gdf_hex["type_diversity"] = gdf_hex["type_diversity"].fillna(0).astype(int)

print(f"Master hex GeoDataFrame shape: {gdf_hex.shape}")
gdf_hex.head()

Master hex GeoDataFrame shape: (997, 9)


,h3_index,catchment_store_count,catchment_type_diversity,is_food_desert,geometry,store_count,shop_types,type_diversity,store_names
0,893944600c7ffff,28,5,False,"POLYGON ((2.1265 41.40591, 2.12683 41.40413, 2...",0,NaN,0,NaN
1,8939446352bffff,1,1,False,"POLYGON ((2.07891 41.42848, 2.07924 41.4267, 2...",0,NaN,0,NaN
2,89394460e67ffff,61,5,False,"POLYGON ((2.18613 41.38522, 2.18646 41.38345, ...",0,NaN,0,NaN
3,8939446017bffff,241,6,False,"POLYGON ((2.16059 41.40037, 2.16092 41.3986, 2...",15,"[supermarket, convenience, bakery, butcher]",4,"[Condis, Supermercat, Pim Pam, Forn Sant Honor..."
4,89394461c07ffff,33,5,False,"POLYGON ((2.14394 41.36751, 2.14427 41.36573, ...",0,NaN,0,NaN


In [9]:
# Spatial join hex centroids → barris
gdf_hex["centroid"] = gdf_hex.geometry.centroid
gdf_hex_centroids = gpd.GeoDataFrame(gdf_hex[["h3_index", "centroid"]], 
                                      geometry="centroid", crs="EPSG:4326")

# Find which column has barri name — print it from Cell 2 output and adjust if needed
NAME_COL = [c for c in gdf_barris.columns if any(x in c.lower() for x in ["nom", "name", "barri"])][0]
print(f"Using neighbourhood name column: '{NAME_COL}'")

gdf_hex = gdf_hex.drop(columns=["centroid"])
gdf_hex = gpd.sjoin(
    gdf_hex,
    gdf_barris[["geometry", NAME_COL]].rename(columns={NAME_COL: "barri_name"}),
    how="left",
    predicate="intersects"
).drop(columns=["index_right"])

print(f"Hexes with barri assigned: {gdf_hex['barri_name'].notna().sum()}")

Using neighbourhood name column: 'BARRI'
Hexes with barri assigned: 1620


In [10]:
# Drop non-serialisable list columns before saving
gdf_hex_save = gdf_hex.drop(columns=["shop_types", "store_names"], errors="ignore")
gdf_hex_save.to_file("../data/processed/barcelona_hex_grid.geojson", driver="GeoJSON")

gdf_pois_clipped.to_file("../data/processed/barcelona_pois_clipped.geojson", driver="GeoJSON")

print("Saved:")
print(f"  → data/processed/barcelona_hex_grid.geojson  ({len(gdf_hex_save)} hexes)")
print(f"  → data/processed/barcelona_pois_clipped.geojson  ({len(gdf_pois_clipped)} POIs)")

Saved:
  → data/processed/barcelona_hex_grid.geojson  (1620 hexes)
  → data/processed/barcelona_pois_clipped.geojson  (2560 POIs)


In [11]:
m = folium.Map(location=[41.3874, 2.1686], zoom_start=13, tiles="CartoDB positron")

# Colour hexes: red = food desert, green = well served
for _, row in gdf_hex_save.iterrows():
    color = "#d73027" if row["is_food_desert"] else "#91cf60" if row["catchment_store_count"] > 5 else "#fee08b"
    folium.GeoJson(
        row["geometry"].__geo_interface__,
        style_function=lambda x, c=color: {
            "fillColor": c, "color": "white",
            "weight": 0.3, "fillOpacity": 0.6
        }
    ).add_to(m)

m.save("../outputs/maps/02_hex_food_desert_preview.html")
print("Map saved → outputs/maps/02_hex_food_desert_preview.html")
print("Open it in your browser to explore!")
m

Map saved → outputs/maps/02_hex_food_desert_preview.html
Open it in your browser to explore!
